## Assignment 1
### Task 1.1 Calculate the Clock Granularity of different Python Timers
We start with defining the given "checktick"-function to calculate the clock granularity of the "time.time()" clock.

In [27]:
from timeit import default_timer as timer
import numpy as np
import time

def checktick():
    M = 200
    timesfound = np.empty((M,))
    for i in range(M):
        t1 =  time.time() # get timestamp from timer
        t2 = time.time() # get timestamp from timer
        while (t2 - t1) < 1e-16: # if zero then we are below clock granularity, retake timing
            t2 = time.time() # get timestamp from timer
        t1 = t2 # this is outside the loop
        timesfound[i] = t1 # record the time stamp
    minDelta = 1000000
    Delta = np.diff(timesfound) # it should be cast to int only when needed
    minDelta = Delta.min() * 1e9 # convert to nanoseconds
    return minDelta
print("Clock granularity of time.time() is ", checktick(), " nanoseconds")

Clock granularity of time.time() is  715.2557373046875  nanoseconds


Now we modify the "checktick"-function to calculate the clock granularity of "timeit".

In [28]:
def checktick():
    M = 200
    timesfound = np.empty((M,))
    for i in range(M):
        t1 =  timer() # get timestamp from timer
        t2 = timer() # get timestamp from timer
        while (t2 - t1) < 1e-16: # if zero then we are below clock granularity, retake timing
            t2 = timer() # get timestamp from timer
        t1 = t2 # this is outside the loop
        timesfound[i] = t1 # record the time stamp
    minDelta = 1000000
    Delta = np.diff(timesfound) # it should be cast to int only when needed
    minDelta = Delta.min() * 1e9 # convert to nanoseconds
    return minDelta
print("Clock granularity of timeit is ", checktick(), " nanoseconds")

Clock granularity of timeit is  165.99369700998068  nanoseconds


Now we modify it for the "time.time_ns" timer:

In [29]:
def checktick():
    M = 200
    timesfound = np.empty((M,))
    for i in range(M):
        t1 =  time.time_ns() # get timestamp from timer
        t2 = time.time_ns() # get timestamp from timer
        while (t2 - t1) < 1e-16: # if zero then we are below clock granularity, retake timing
            t2 = time.time_ns() # get timestamp from timer
        t1 = t2 # this is outside the loop
        timesfound[i] = t1 # record the time stamp
    minDelta = 1000000
    Delta = np.diff(timesfound) # it should be cast to int only when needed
    minDelta = Delta.min()
    return minDelta
print("Clock granularity of time.time_ns is ", checktick(), " nanoseconds")

Clock granularity of time.time_ns is  768.0  nanoseconds


The results show, that the "timeit" timer has the smallest granularity.

### Task 1.2 Timing the Julia set code functions.

We modified the given decorator and put it in a new file, to keep the original one. The new code is in the file namend "JuliaSet-timing".
Changes made to the decorator:
- Change the timer to the one from timeit, because it turned out to have the smallest clock granularity.
- Run Code multiple times (5 times), to get average and standard deviation data.

Modified code of the decorator:
```python
def timefn(fn):
    @wraps(fn)
    def measure_time(*args, **kwargs):
        n = 5
        times = []

        for i in range(n):
            t1 = timer()
            result = fn(*args, **kwargs)
            t2 = timer()
            times.append(t2 - t1)
        avg = np.mean(times)
        std = np.std(times)
        print(f"@timefn: {fn.__name__} took {avg:.6f} seconds, standard deviation: {std:.6f} (n={n})")
        return result
    return measure_time
```
Since the "calc_pure_python" function calls the "calculate_z_serial_purepython" inside itself, we first only timed the "calc_pure_python" function (in JuliaSet-timing.py).

In [30]:
%run JuliaSet-timing.py

Length of x: 1000
Total elements: 1000000
Length of x: 1000
Total elements: 1000000
Length of x: 1000
Total elements: 1000000
Length of x: 1000
Total elements: 1000000
Length of x: 1000
Total elements: 1000000
@timefn: calc_pure_python took 3.963043 seconds, standard deviation: 0.042175 (n=5)


Now the function "calculate_z_serial_purepython" is timed, for that we use the exact same code as before but only with the decorator used on "calculate_z_serial_purepython".
This is done in a separate file "JuliaSet-timing2.py".

In [31]:
%run JuliaSet-timing2.py

Length of x: 1000
Total elements: 1000000
@timefn: calculate_z_serial_purepython took 3.647499 seconds, standard deviation: 0.026005 (n=5)


Since the standard deviation is far bigger than the clock granularity, the timer function is not limited by the clock granularity. If it was the same value as the clock granularity, this would mean that the timer is not fast enough and the results depend on the timer itself.

### Task 1.3 Profile the Julia set code with cProfile and line_profiler the computation.

Since "cProfile" doesn't need changes in the code we can use the original code in the file "JuliaSet-original.py". We will time the execution with the profilers to analyze the overhead afterwards.

In [32]:
t1 =  timer()
!python -m cProfile -s cumulative JuliaSet-original.py
t2 = timer()
time_cProfiler = t2 - t1
print("Execution time with cProfile: ", time_cProfiler, " seconds")

Length of x: 1000
Total elements: 1000000
calculate_z_serial_purepython took 4.953953981399536 seconds
         36221995 function calls in 5.266 seconds

   Ordered by: cumulative time

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        1    0.000    0.000    5.266    5.266 {built-in method builtins.exec}
        1    0.011    0.011    5.266    5.266 JuliaSet-original.py:1(<module>)
        1    0.250    0.250    5.255    5.255 JuliaSet-original.py:21(calc_pure_python)
        1    4.005    4.005    4.954    4.954 JuliaSet-original.py:59(calculate_z_serial_purepython)
 34219980    0.948    0.000    0.948    0.000 {built-in method builtins.abs}
  2002000    0.047    0.000    0.047    0.000 {method 'append' of 'list' objects}
        1    0.004    0.004    0.004    0.004 {built-in method builtins.sum}
        3    0.000    0.000    0.000    0.000 {built-in method builtins.print}
        2    0.000    0.000    0.000    0.000 {built-in method time.time}
       

Now let's save it to a file so we can visualize the results with snakeviz afterwards.

In [33]:
!python -m cProfile -o profile.stats JuliaSet-original.py

Length of x: 1000
Total elements: 1000000
calculate_z_serial_purepython took 4.923141002655029 seconds


In [34]:
!python -m snakeviz profile.stats --server

snakeviz web server started on 127.0.0.1:8080; enter Ctrl-C to exit
http://127.0.0.1:8080/snakeviz/%2FUsers%2Ftimeberle%2FGitHub%2FHPC-Intro%2Fprofile.stats
^C


First we will use the lineProfiler for the first function "calc_pure_python". Therefore we need to use the decorator only on this function, which is done in the file "JuliaSet-lineProfiler.py".

In [35]:
t1 =  timer()
!python -m kernprof -l JuliaSet-lineProfiler.py
t2 =  timer()
!python -m line_profiler JuliaSet-lineProfiler.py.lprof
time_lineProfiler1 = t2 - t1
print("Execution time with lineProfiler on calc_pure_python: ", time_lineProfiler1, " seconds")

Length of x: 1000
Total elements: 1000000
calculate_z_serial_purepython took 14.613469123840332 seconds
Wrote profile results to 'JuliaSet-lineProfiler.py.lprof'
Inspect results with:
python -m line_profiler -rmt JuliaSet-lineProfiler.py.lprof
Timer unit: 1e-06 s

Total time: 15.3213 s
File: JuliaSet-lineProfiler.py
Function: calc_pure_python at line 20

Line #      Hits         Time  Per Hit   % Time  Line Contents
    20                                           @profile
    21                                           def calc_pure_python(desired_width, max_iterations):
    22                                               """Create a list of complex coordinates (zs) and complex parameters (cs),
    23                                               build Julia set"""
    24         1          2.0      2.0      0.0      x_step = (x2 - x1) / desired_width
    25         1          0.0      0.0      0.0      y_step = (y1 - y2) / desired_width
    26         1          0.0      0.0      0

Now let's do it also for the function "calculate_z_serial_purepython", in file "JuliaSet-lineProfiler2.py".

In [36]:
t1 =  timer()
!python -m kernprof -l JuliaSet-lineProfiler2.py
t2 =  timer()
!python -m line_profiler JuliaSet-lineProfiler2.py.lprof
time_lineProfiler2 = t2 - t1
print("Execution time with lineProfiler on calculate_z_serial_purepython: ", time_lineProfiler2, " seconds")

Length of x: 1000
Total elements: 1000000
calculate_z_serial_purepython took 31.534297943115234 seconds
Wrote profile results to 'JuliaSet-lineProfiler2.py.lprof'
Inspect results with:
python -m line_profiler -rmt JuliaSet-lineProfiler2.py.lprof
Timer unit: 1e-06 s

Total time: 19.8073 s
File: JuliaSet-lineProfiler2.py
Function: calculate_z_serial_purepython at line 58

Line #      Hits         Time  Per Hit   % Time  Line Contents
    58                                           @profile
    59                                           def calculate_z_serial_purepython(maxiter, zs, cs):
    60                                               """Calculate output list using Julia update rule"""
    61         1        664.0    664.0      0.0      output = [0] * len(zs)
    62   1000001     183829.0      0.2      0.9      for i in range(len(zs)):
    63   1000000     138619.0      0.1      0.7          n = 0
    64   1000000     184443.0      0.2      0.9          z = zs[i]
    65   1000000

To analyze the overhead of the profilers we will compare the time it takes to run the code with and without the profilers. Therefore we will run it once without the profilers.

In [37]:
t1 =  timer()
%run JuliaSet-original.py
t2 = timer()
time_noProfiler = t2 - t1
print("Execution time without profiler: ", time_noProfiler, " seconds")
overhead_cProfiler = time_cProfiler - time_noProfiler
overhead_lineProfiler1 = time_lineProfiler1 - time_noProfiler
overhead_lineProfiler2 = time_lineProfiler2 - time_noProfiler
print("Overhead of cProfile: ", overhead_cProfiler, " seconds")
print("Overhead of lineProfiler on calc_pure_python: ", overhead_lineProfiler1, " seconds")
print("Overhead of lineProfiler on calculate_z_serial_purepython: ", overhead_lineProfiler2, " seconds")

Length of x: 1000
Total elements: 1000000
calculate_z_serial_purepython took 3.6402270793914795 seconds
Execution time without profiler:  3.865267250002944  seconds
Overhead of cProfile:  1.72140279100131  seconds
Overhead of lineProfiler on calc_pure_python:  12.172126707999269  seconds
Overhead of lineProfiler on calculate_z_serial_purepython:  28.27802058299858  seconds


### Task 1.4 Memory-profile the Juliaset code. Use the memory_profiler and mprof to profile the computation in JuliaSet code.

Now we will profile the memory usage of the JuliaSet code. We start with using the memory_profiler on the two functions. We will also time the execution times to measure the overhead afterwards.
Since it needs the same decorator as the lineprofiler we can use the same files for it.

In [38]:
t1 =  timer()
!python -m memory_profiler JuliaSet-lineProfiler.py
t2 =  timer()
time_memory_profiler1 = t2 - t1
print("Execution time with memory_profiler on calc_pure_python: ", time_memory_profiler1, " seconds")
t1 =  timer()
!python -m memory_profiler JuliaSet-lineProfiler2.py
t2 =  timer()
time_memory_profiler2 = t2 - t1
print("Execution time with memory_profiler on calculate_z_serial_purepython: ", time_memory_profiler2, " seconds")

Length of x: 1000
Total elements: 1000000
calculate_z_serial_purepython took 47.21134901046753 seconds
Filename: JuliaSet-lineProfiler.py

Line #    Mem usage    Increment  Occurrences   Line Contents
    20   44.766 MiB   44.766 MiB           1   @profile
    21                                         def calc_pure_python(desired_width, max_iterations):
    22                                             """Create a list of complex coordinates (zs) and complex parameters (cs),
    23                                             build Julia set"""
    24   44.766 MiB    0.000 MiB           1       x_step = (x2 - x1) / desired_width
    25   44.766 MiB    0.000 MiB           1       y_step = (y1 - y2) / desired_width
    26   44.766 MiB    0.000 MiB           1       x = []
    27   44.766 MiB    0.000 MiB           1       y = []
    28   44.766 MiB    0.000 MiB           1       ycoord = y2
    29   44.781 MiB    0.016 MiB        1001       while ycoord > y1:
    30   44.781 MiB    0.00

In [44]:
t1 =  timer()
!python -m mprof run JuliaSet-lineProfiler.py
t2 =  timer()
time_mprof1 = t2 - t1
print("Execution time with mprof on calc_pure_python: ", time_mprof1, " seconds")
t1 =  timer()
!python -m mprof run JuliaSet-lineProfiler2.py
t2 =  timer()
time_mprof2 = t2 - t1
print("Execution time with mprof on calculate_z_serial_purepython: ", time_mprof2, " seconds")

mprof.py: Sampling memory every 0.1s
running new process
running as a Python program...
Length of x: 1000
Total elements: 1000000
calculate_z_serial_purepython took 3.6699697971343994 seconds
Execution time with mprof on calc_pure_python:  4.858620666003844  seconds
mprof.py: Sampling memory every 0.1s
running new process
running as a Python program...
Length of x: 1000
Total elements: 1000000
calculate_z_serial_purepython took 3.71606183052063 seconds
Execution time with mprof on calculate_z_serial_purepython:  4.78319695800019  seconds


The mprof plots can't be displayed when executing in Jupyter Notebook. The following two images are the resulting plots, first of only the calc_pure_python, second of calculate_z_serial_purepython function:
![Plot of calc_pure_python mprof results](mprof_plot_task1_4-1.png)
![Plot of calculate_z_serial_purepython mprof results](mprof_plot_task1_4-2.png)

In [45]:
overhead_mprof1 = time_mprof1 - time_noProfiler
overhead_mprof2 = time_mprof2 - time_noProfiler
print("Overhead of mprof on calc_pure_python: ", overhead_mprof1, " seconds")
print("Overhead of mprof on calculate_z_serial_purepython: ", overhead_mprof2, " seconds")

Overhead of mprof on calc_pure_python:  0.9933534160009003  seconds
Overhead of mprof on calculate_z_serial_purepython:  0.9179297079972457  seconds
